<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  8. Agents
</h2>

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  8.1 What Are Agents?
</h3>

**Reminders from previous notebooks**

When you start a new **Conversation**, you have to choose whether:
- You configure all parameters such as `model`, `instructions` and `tools` in the **Conversation**
- Or you configure these parameters in an [**Agent**](https://docs.mistral.ai/studio-api/agents/agents-api#agents) which can be linked to a new conversation via the agent ID

As a result:
- Your **Conversation** must use a `model` OR an `agent_id` parameter. Setting both returns an error.
- In case your **Conversation** is configured via an **Agent**, adding the `instructions` or `tools` parameters in the **Conversation** would return an error

<div style="background-color: #FFF3E0; padding: 12px; border-left: 4px solid #FF8F00; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #9F521A;">Key Takeaway:</strong>
  The Agent is an optional piece of re-usable configuration
</div>

But why use an **Agent** if **stand-alone Conversations** have nearly the same capabilities?

| Capability | With Agent | Without Agent |
| --- | --- | --- |
| Reusable configuration | ✅ | ❌ |
| Updatable configuration during a conversation | ✅ | ❌ |
| Multi-agents capabilities (handoffs) | ✅ | ❌ |

In [1]:
from mistralai.client import Mistral
from mistralai.client.models import UserMessage
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]
mistral = Mistral(api_key=api_key)

In [17]:
from mistralai.client.models import MessageOutputEntry, ToolExecutionEntry, TextChunk, ToolReferenceChunk, ToolFileChunk, AgentHandoffEntry
from IPython.display import display, Markdown, Image
import json

# Taken from the previous notebook and added the displayed_agents parameter + some support for handoff entries
def display_entries(entries: list, displayed_agents: list[str] | None = None, download_files: bool = False) -> None:
    """
    Function to display entries in a controlled manner

    args:
      entries: list
          Conversation entries from the ConversationResponse.outputs list
      displayed_agents: list | None (default=None)
          List of agent IDs for which to display output messages.
          Add your main agent so that intermediary messages from
          other subagents are not displayed.
      download_files: bool (default = False)
          Whether to download in the cwd files found in the MessageOutputEntry objects
          May be useful as the LLM sometimes assumes that these files are available
          and tries to display them.
    """

    # These lists are used to collect various outputs before we decide in which order to display them 
    assistant_list = []
    additional_content = []
    handoffs = []

    # We use a set for references to avoid duplicates and because the order is not important
    references = set()

    # Loop through all entries 
    for entry in entries:

        # OUTPUT MESSAGES ----------------------------------------------------------------------------------------------
        if isinstance(entry, MessageOutputEntry):

            if displayed_agents is None or entry.agent_id in displayed_agents:

                # If the message content is a single string, we just display it
                if isinstance(entry.content, str) and entry.content!=".":
                    assistant_list.append(Markdown(entry.content))
                    continue

                # If the message content consists of multiple content chunks, we need to loop through them
                for chunk in entry.content:
                    if isinstance(chunk, TextChunk) and chunk.text!=".":
                        assistant_list.append(Markdown(chunk.text))
                    elif isinstance(chunk, ToolReferenceChunk):
                        match chunk.tool:
                            case "web_search":
                                references.add(f"[{chunk.title}]({chunk.url})<br>")
                            case "document_library":
                                references.add(f"[{chunk.title}](https://chat.mistral.ai/libraries/019e93b2-8cd4-76a4-bded-ebe721d359a5?document={chunk.url})<br>")
                            case "code_interpreter":
                                pass
                    elif isinstance(chunk, ToolFileChunk) and download_files==True:
                        # Some images had a filename without extension
                        extension = f".{chunk.file_type}"
                        if not chunk.file_name.endswith(extension):
                            filename = f"{chunk.file_name}{extension}"
                        else:
                            filename = chunk.file_name
                        # Download file in current working directory (useful when the LLM tries to display files but only provides the filename)
                        file_content = mistral.files.download(file_id=chunk.file_id).read() 
                        with open(f"{filename}", "wb") as f:
                            f.write(file_content)
                    
        
        # TOOL EXECUTION MESSAGES --------------------------------------------------------------------------------------
        elif isinstance(entry, ToolExecutionEntry):

            match entry.name:
                case "web_search":
                    try:
                        query = json.loads(entry.arguments)["query"]
                        display(Markdown(f"*⏳ Searching the web for `{query}`...*"))
                    except:
                        pass
                case "code_interpreter":
                    additional_content.append(Markdown(f"```text {entry.info["code"]}"))
                    results = entry.info["result"]
                    for result in results:
                        if result["type"]=="file_url":
                            additional_content.append(Image(url=result["file_url"]))
                case "image_generation":
                    additional_content.append(Image(url=json.loads(entry.info["result"])["url"]))
        
        # HANDOFFS -----------------------------------------------------------------------------------------------------
        elif isinstance(entry, AgentHandoffEntry):
            handoffs.append(f"Handoff {entry.previous_agent_name} => {entry.next_agent_name}")
    
    # Display the assistant messages on top
    display(Markdown(f"**🤖 ASSISTANT** {'-' * 100}"))
    for content in assistant_list:
        display(content)

    # Continue with additional content to show in the conversation (images, code samples, charts)
    for content in additional_content:
        display(content)
    
    # Finish with references/citations if any.
    # Only strings are stored in the set so we can avoid duplicates (saving Markdown() objects doesn't work as each object is different even if their representation is the same)
    if references:
        display(Markdown(f"**📚 REFERENCES** {'-' * 100}"))
        for content in references:
            display(Markdown(content))
    
    # NEW - List of all handoffs
    if handoffs:
        display(Markdown(f"**🤝 HANDOFFS** {'-' * 100}"))
        for content in handoffs:
            print(content)

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  8.2 Connect an Agent to a Conversation
</h3>

In [19]:
from mistralai.client.models import WebSearchTool

# Re-used an example from a previous notebook in which we connected the web search tool to a conversation
# All configuration items are moved to an agent now
journalist = mistral.beta.agents.create(
    model="mistral-medium-latest",
    name="Journalist",
    description="an agent having access to a web search tool",
    instructions="You are a journalist looking for fresh news with your web_search tool",
    tools=[WebSearchTool()],
    completion_args={"temperature": 0.7, "top_p": 1}
)

journalist

Agent(model='mistral-medium-latest', name='Journalist', id='ag_019ed69713e87037b8bbbfc882a7559e', version=0, versions=[], created_at=datetime.datetime(2026, 6, 17, 17, 18, 5, 827263, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 6, 17, 17, 18, 5, 827266, tzinfo=TzInfo(0)), deployment_chat=False, source='api', instructions='You are a journalist looking for fresh news with your web_search tool', tools=[WebSearchTool(tool_configuration=None, type='web_search')], completion_args=CompletionArgs(stop=None, presence_penalty=None, frequency_penalty=None, temperature=0.7, top_p=1.0, max_tokens=None, random_seed=None, prediction=None, response_format=None, tool_choice='auto', reasoning_effort=None), guardrails=[], description='an agent having access to a web search tool', handoffs=None, metadata=None, object='agent', version_message=None)

In [4]:
# When we instantiate the conversation, we link it to an agent via ID, instead of providing a model name
response = mistral.beta.conversations.start(
    agent_id=journalist.id,
    inputs="What is the biggest financial news that happened on June 12th 2026?"
)

display_entries(response.outputs)

*⏳ Searching the web for `biggest financial news June 12 2026`...*

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

The biggest financial news on June 12, 2026, was SpaceX's historic initial public offering (IPO). SpaceX raised approximately $75 billion, achieving an initial valuation near $1.8 trillion, making it the largest IPO in history. The company began trading on the Nasdaq, and its stock surged by about 20% on its debut, further boosting market sentiment. This event, combined with hopes of a potential U.S.-Iran peace deal, led to significant gains in global equities and a sharp drop in oil prices, as Brent crude fell over 3% to around $87.25 per barrel

**📚 REFERENCES** ----------------------------------------------------------------------------------------------------

[Investment Update on Financial Market News – 12 June 2026 — Patronus Partners](https://www.patronuspartners.com/daily-insights/jls3z7jejwezwra5dpttey673cc4pp)<br>

[The Week That Was: June 12, 2026](https://www.cnbc.com/video/2026/06/12/the-week-that-was-june-12-2026.html)<br>

[Stock Market News — June 12, 2026 — Morning Update — Last 12 Hours (Pacific Time)](https://www.everhint.com/stock-market-news-june-12-2026-morning-update-last-12-hours-pacific-time/)<br>

[Stock market news for June 12, 2026](https://www.cnbc.com/2026/06/11/stock-market-today-live-updates.html)<br>

[5 Things to Know Before the Stock Market Opens](https://www.investopedia.com/5-things-to-know-before-the-stock-market-opens-june-12-2026-11996466)<br>

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  8.3 Update an Agent
</h3>

In [10]:
updated_journalist = mistral.beta.agents.update(
    agent_id=journalist.id,
    instructions="Always respond with A LOT of emojis"
)

updated_journalist

Agent(model='mistral-medium-latest', name='Journalist', id='ag_019ed68a73e371ffae33c7ad287ccb1b', version=2, versions=[], created_at=datetime.datetime(2026, 6, 17, 17, 4, 18, 434698, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 6, 17, 17, 7, 2, 555580, tzinfo=TzInfo(0)), deployment_chat=False, source='api', instructions='Always respond with A LOT of emojis', tools=[WebSearchTool(tool_configuration=None, type='web_search')], completion_args=CompletionArgs(stop=None, presence_penalty=None, frequency_penalty=None, temperature=None, top_p=None, max_tokens=None, random_seed=None, prediction=None, response_format=None, tool_choice='auto', reasoning_effort=None), guardrails=[], description='an agent having access to a web search tool', handoffs=None, metadata=None, object='agent', version_message=None)

In [11]:
mistral.beta.agents.get(
    agent_id=journalist.id,
    agent_version=1
).instructions

'Always respond with a lot of emojis'

In [12]:
response = mistral.beta.conversations.start(
    agent_id=journalist.id,
    inputs="What happened during the G7 2026 so far?"
)

display_entries(response.outputs)

*⏳ Searching the web for `G7 2026 summit updates and outcomes so far`...*

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

Here’s what’s happened so far at the **G7 2026 Summit** in Evian-les-Bains, France (June 15–17, 2026) 🌍✨:

---

**🔥 Major Highlights:**

- **US-Iran Peace Deal:** G7 leaders **welcomed the historic US-Iran agreement** aimed at ending the West Asia conflict, calling it a “historic opportunity” to restore regional stability. They also discussed the swift reopening of the **Strait of Hormuz** and alternative energy routes to bypass the waterway 🚢💥

.

- **Ukraine Support:** The G7 **agreed to intensify pressure on Russia** to end its war in Ukraine, with new pledges to strengthen Ukraine’s air defense and increase diplomatic and economic pressure on Moscow. Ukrainian President **Volodymyr Zelenskyy** attended the summit and secured important commitments from G7 leaders for further support 🇺🇦🤝

.

- **AI & Critical Minerals:** Leaders discussed **granting “trusted partners” access to cutting-edge US AI models** and tackling reliance on China for critical minerals, with France pushing for measures to reduce dependence and boost economic sovereignty 🤖💻

...

[Trump projects confidence about Iran deal as he meets global leaders for G7 summit](https://apnews.com/live/g7-summit-trump-updates-06-15-2026)<br>

[G7 Summit Highlights: Successful partnerships are built on trust, PM Modi tells world leaders](https://www.indiatoday.in/amp/world/story/g7-summit-in-france-2026-live-updates-pm-modi-donald-trump-meeting-2927735-2026-06-16)<br>

[Trump says he will push for peace in Ukraine after meeting Zelenskyy at G7 | European Union News | Al Jazeera](https://www.aljazeera.com/news/2026/6/16/g7-leaders-meet-in-france-with-iran-and-ukraine-high-on-agenda)<br>

[G7 Summit 2026 Live Updates: 'India will play a big role in West Asia under PM Modi,' says Trump – Firstpost](https://www.firstpost.com/world/g7-summit-2026-live-updates-pm-modi-in-france-donald-trump-bilateral-meeting-us-iran-war-ukraine-macron-france-evian-keir-starmer-2-liveblog-14023271.html)<br>

[G7 Summit Day 1 highlights: PM Modi voices concern on maritime trade impact due to Hormuz disruptions, says several Indians lost their lives during West Asia conflict - The Hindu](https://www.thehindu.com/news/international/g7-summit-2026-france-day-1-leaders-discuss-west-asia-conflict-ukraine-russia-war-live-updates-june-16-2026/article71107789.ece)<br>

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  8.4 Handoffs (Multi-Agents)
</h3>

The main benefit of a multiagents setup is specialization and context isolation.

In [13]:
from mistralai.client.models import ImageGenerationTool

# Create new agent: Artist
artist = mistral.beta.agents.create(
    model="mistral-medium-latest",
    name="Artist",
    description="an agent with creative skills to generate images",
    instructions="You are an artist in charge of generating images of recent events",
    tools=[ImageGenerationTool()]
)

# Create new agent: Manager
manager = mistral.beta.agents.create(
    model="mistral-medium-latest",
    name="Manager",
    description="the manager is the orchestrator: dispatching tasks, gathering intermediate results and producing the final answer",
    instructions="""
    You are in charge of planning a task and delegating steps to various agents.
    When you receive the result back, you may still delegate a new task to another agent.
    Don't respond until all tasks are performed.
    Use the artist agent for generating images, or the journalist to get internet news.
    Your final response should include all details as end-users won't see intermediate steps from other agents
    """
)

# Add the handoff journalist => manager
mistral.beta.agents.update(
    agent_id=journalist.id,
    handoffs=[manager.id],
    instructions="ALWAYS return the conversation to your manager once you're done with your task, don't respond to the end user"
)

# Add the handoff artist => manager
mistral.beta.agents.update(
    agent_id=artist.id,
    handoffs=[manager.id],
    instructions="ALWAYS return the conversation to your manager once you're done with your task, don't respond to the end user"
)

# Add the handoff manager => journalist/artist
mistral.beta.agents.update(
    agent_id=manager.id,
    handoffs=[journalist.id, artist.id]
)

Agent(model='mistral-medium-latest', name='Manager', id='ag_019ed68f501674ffa83e76a0ba13d54c', version=1, versions=[], created_at=datetime.datetime(2026, 6, 17, 17, 9, 36, 944123, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 6, 17, 17, 9, 37, 577548, tzinfo=TzInfo(0)), deployment_chat=False, source='api', instructions="\n    You are in charge of planning a task and delegating steps to various agents.\n    When you receive the result back, you may still delegate a new task to another agent.\n    Don't respond until all tasks are performed.\n    Use the artist agent for generating images, or the journalist to get internet news.\n    Your final response should include all details as end-users won't see intermediate steps from other agents\n    ", tools=[], completion_args=CompletionArgs(stop=None, presence_penalty=None, frequency_penalty=None, temperature=None, top_p=None, max_tokens=None, random_seed=None, prediction=None, response_format=None, tool_choice='auto', reasoning_effo

In [14]:
response = mistral.beta.conversations.start(
    agent_id=manager.id,
    inputs="What are keys news from the last Mistral AI Summit? Generate an image to illustrate the response.",
    handoff_execution="server" # Optional, default is "server". "client" allows you to get back control at each handoff
)

In [15]:
response.outputs

[AgentHandoffEntry(previous_agent_id='ag_019ed68f501674ffa83e76a0ba13d54c', previous_agent_name='Manager', next_agent_id='ag_019ed68a73e371ffae33c7ad287ccb1b', next_agent_name='Journalist', object='entry', type='agent.handoff', created_at=datetime.datetime(2026, 6, 17, 17, 10, 19, 584576, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 17, 17, 10, 19, 584635, tzinfo=TzInfo(0)), id='handoff_019ed68ff6c072a4baa2467e93140a66'),
 ToolExecutionEntry(name='web_search', arguments='{"query": "Mistral AI Summit 2026 key news", "start_date": "2026-06-01", "end_date": "2026-06-17", "limit": 10}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 6, 17, 17, 10, 20, 727814, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 6, 17, 17, 10, 22, 799646, tzinfo=TzInfo(0)), agent_id='ag_019ed68a73e371ffae33c7ad287ccb1b', model='mistral-medium-latest', id='tool_exec_019ed68ffb37726ba97f8f08e938f304', info={'result': '{"0": {"url": "https://mistral.ai/news/ai-now-su

In [18]:
display_entries(response.outputs, displayed_agents=[manager.id])

*⏳ Searching the web for `Mistral AI Summit 2026 key news`...*

**🤖 ASSISTANT** ----------------------------------------------------------------------------------------------------

Here are the key news highlights from the **Mistral AI Summit 2026**, held in Paris:

---

### **1. Mistral for Industrial Engineering**
Mistral AI unveiled an **integrated AI stack** combining advanced physics models, engineering expertise, and robotics to revolutionize industrial operations. This solution aims to:
- Accelerate design processes and eliminate simulation bottlenecks.
- Optimize asset performance while maintaining full control over proprietary data and IP.
- **Partnerships**:
  - **Airbus**: AI will be embedded in operations from design to on-board capabilities, enhancing flight safety and innovation.
  - **BMW Group**: Collaboration on the **"Large Industry Model" (LIM)** initiative to build multimodal reasoning models for complex use cases like crash simulations.
  - **ASML**: AI-driven optimization for semiconductor manufacturing, including high-performance part design and control loops.
- **Acquisition of Emmi AI**: Mistral acquired Emmi to enhance its physics AI capabilities, enabling faster design, simulation, and production in aerospace, automotive, and semiconductor industries.

---

### **2. Vibe: A Unified Agent for Long-Horizon Productivity**
Mistral introduced **Vibe**, an AI agent designed for long-running, multi-step tasks:
- Manages emails, calendars, and deep research.
- Drafts deliverables and automates recurring processes.
- **Coding Capabilities**: Builds features, fixes bugs, refactors code, and ships pull requests across web apps, editors, and terminals.
- Runs on Mistral’s flagship models optimized for reasoning, agentic tasks, and coding.

---

### **3. Les Ulis Data Center**
- A **new 10 MW facility** in Les Ulis (Essonne, France) dedicated to inference operations.
- Scheduled to open in **Q3 2026**, it will address compute supply chain risks by providing direct control over capacity, enhancing security and transparency.

---
---

### **4. Funding and Valuation**
- Mistral AI is in talks to raise **€3 billion (≈$3.5 billion)** at a valuation of **€20 billion (≈$23.1 billion)**, nearly double its previous valuation of €11.7 billion from September 2025.
- The company now employs **1,000 people** and is targeting **€1 billion in revenue for 2026**.
- Mistral is positioning itself as a **European alternative** to OpenAI and Anthropic, focusing on sovereign AI infrastructure for governments and enterprises.

---
---
### **5. Strategic Vision**
Mistral AI is expanding its focus on:
- **Full-stack AI solutions** for enterprises and governments.
- **Secure infrastructure** to ensure data sovereignty and control.
- **Industrial AI** to solve complex engineering challenges in aerospace, automotive, and semiconductors.

---
---
### **Illustration of the Summit**
Below is an image illustrating the Mistral AI Summit 2026, capturing the futuristic and innovative atmosphere of the event:

![Mistral AI Summit 2026](https://mistralaiblackforestprod.blob.core.windows.net/images/blackforest/c55f/d729/-675/5-428f-97da-a8b0cbec04ab/image.jpg?se=2026-06-17T18%3A11%3A03Z&sp=r&sv=2026-02-06&sr=b&skoid=8aae9820-8683-45ec-b557-e441def5aa94&sktid=4fbc1168-2984-4d17-af19-ac5138c2378e&skt=2026-06-17T17%3A11%3A03Z&ske=2026-06-17T18%3A11%3A03Z&sks=b&skv=2026-02-06&sig=dLwmOEcdtz3OKgCy3p/u7vv2uO8PYXY/j4Jp5tQ9zYQ%3D)

---
Would you like more details on any specific announcement or partnership?

**🤝 HANDOFFS** ----------------------------------------------------------------------------------------------------

Handoff Manager => Journalist
Handoff Journalist => Manager
Handoff Manager => Artist
Handoff Artist => Manager


<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  8.5 To go further
</h3>

You can also review your agents here: https://console.mistral.ai/build/agents?source=api <br>
You can also deploy them in Vibe

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  8.6 Resources Clean-Up
</h3>

In [20]:
for agent in mistral.beta.agents.list():
    id = agent.id
    mistral.beta.agents.delete(agent_id=id)

mistral.beta.agents.list()

[]

In [21]:
for conv in mistral.beta.conversations.list():
    mistral.beta.conversations.delete(conversation_id=conv.id)

mistral.beta.conversations.list()

[]

In [22]:
for file in mistral.files.list().data:
    mistral.files.delete(file_id=file.id)

mistral.files.list().data

[]